## Executive Summary

**Purpose:** the reference page for the `emachines.inductance` module. It explains what the inductance of a winding is, which flux paths make it up, how the phase values relate to the synchronous and dq inductances used in motor models, and what an inductance measurement actually contains. Every part is mapped to the notebook and function that computes it.

**Reading order:** this page → [Stator and Slot Geometry](inductance_geometry.html) (the inputs) → [Air-Gap Inductance](inductance_airgap.html) (the first part computed) → the leakage notebooks as they are added.

**Assumptions used throughout the module (linear model):** iron of infinite permeability, no saturation, no eddy currents, a smooth (non-salient) rotor. Saturation is a later MEC step.

## 1. What the inductance of a winding is

**Flux linkage.** A coil of $N$ turns, each turn enclosing flux $\Phi_k$, links

$$\lambda = \sum_{k=1}^{N} \Phi_k \quad (= N\Phi \text{ when every turn encloses the same flux}).$$

The voltage across the coil is $v = R\,i + d\lambda/dt$. In a distributed winding different turns enclose different amounts of flux, because they sit in different slots and span different teeth. Counting that correctly is the job of the *winding function* (see [Air-Gap Inductance](inductance_airgap.html)).

**Inductance.** In a linear magnetic circuit the flux is proportional to the current, so

$$L = \frac{\lambda}{i} = N^2\,\mathcal{P}, \qquad W_{mag} = \tfrac12 L i^2,$$

with $\mathcal{P}$ the permeance of the flux path ($\mathcal{P} = \mu_0 A / \ell$ for a uniform air path of area $A$ and length $\ell$). Inductance is therefore *turns squared times the permeance of the path the flux takes*. Since a winding drives flux along several paths in parallel, its inductance is a sum of parts, one per path (section 2).

**Several phases: self and mutual.** With $m$ phases every phase links flux driven by every other phase:

$$\boldsymbol{\lambda} = \mathbf{L}\,\mathbf{i}, \qquad L_{xy} = \left.\frac{\lambda_x}{i_y}\right|_{\text{only } y \text{ excited}}, \qquad W_{mag} = \tfrac12\,\mathbf{i}^T\mathbf{L}\,\mathbf{i}.$$

- $L_{xx}$ (the diagonal) is the **self inductance** of phase $x$: flux of phase $x$ linking phase $x$.
- $L_{xy}$ (off-diagonal) is the **mutual inductance**: flux of phase $y$ linking phase $x$. $L_{xy} = L_{yx}$ (reciprocity), so $\mathbf{L}$ is symmetric.

![Two-pole stator with phase A and phase B coils 120 degrees apart. The flux of a current in phase A crosses the rotor vertically and returns round the yoke. All of it links coil A, giving L_aa; coil B is turned 120 degrees from the flux and links cos 120 = -1/2 of it, so the air-gap mutual inductance is -L_m/2.](images/self_mutual.svg)

For a 3-phase winding the air-gap mutual is negative: the axes are 120° apart, so phase B links $\cos 120^\circ = -\tfrac12$ of the air-gap flux of phase A.

## 2. Where the flux goes: the parts of the inductance

![Unrolled stator and rotor cross-section and a side view. Blue: main flux, down one tooth, across the gap, along the rotor and back up another tooth. Orange: slot leakage, crossing the slot from tooth to tooth and closing through the yoke. Green: tooth-tip leakage, crossing the slot opening between the two tooth tips. Purple: end-winding leakage, circling the end turns outside the stack.](images/flux_paths.svg)

The current in a phase drives flux along four kinds of path. Each has its own permeance, so each gives its own part of the inductance:

| Path (colour above) | Where the flux goes | Part of $L$ | What it depends on |
|---|---|---|---|
| **Main / air-gap** (blue) | down a tooth, across the gap, through the rotor, back up another tooth | $L_{ag} = L_m + L_\delta$ | winding distribution, bore radius, stack length, effective gap |
| **Slot leakage** (orange) | across the slot from tooth to tooth, never reaching the gap | $L_{slot}$ | slot shape, and **where in the slot each conductor sits** |
| **Tooth-tip leakage** (green) | across the slot opening between the tooth tips | $L_{tt}$ | opening width $b_0$, tip height, air gap |
| **End-winding leakage** (purple) | around the end turns, in air outside the stack | $L_{ew}$ | end-turn length and shape, coil pitch |

(A skewed stator or rotor adds a skew leakage; it is not modelled.)

**The air-gap part splits in two.** The flux that crosses the gap is described by the winding function, a staircase round the bore. Its Fourier series has the *working harmonic* $\nu = p$, which matches the rotor's pole count and produces the torque, and every other harmonic:

- **magnetising inductance $L_m$** — the working harmonic only;
- **harmonic (differential) leakage $L_\delta$** — all the other harmonics. This flux does cross the gap, but it has the wrong pole count to make steady torque with the rotor, so as far as the energy conversion goes it is leakage. Distributed windings have little ($L_\delta/L_m \approx 1$–$10\,\%$); tooth-coil windings have a lot (about $100\,\%$ for 12 slots / 10 poles).

**Leakage** is everything that is not the magnetising part:

$$L_\sigma = L_\delta + L_{slot} + L_{tt} + L_{ew}.$$

![Tree: the measured self inductance L_aa splits into the air-gap part (magnetising L_m and harmonic leakage L_delta, both computed in 03_airgap) and local leakage that never reaches the rotor (slot, tooth tip and end winding, computed in later notebooks). Total leakage is L_delta plus slot plus tooth tip plus end winding.](images/inductance_components.svg)

**Why conductor position in the slot matters.** Slot-leakage flux crosses the slot at some depth $y$ and links only the conductors *below* that line (deeper in the slot). A conductor at the slot bottom is linked by flux crossing at every depth; one next to the opening is linked by very little. So layers at different depths have different slot leakage, and phases that share a slot are coupled by it (a mutual slot leakage). That is the source of the layer-to-layer imbalance in windings where each phase sits in its own layer.

## 3. From the phase matrix to the inductances used in motor models

For a balanced 3-phase winding, write the self inductance as $L_{aa}$ and the mutual between any two phases as $L_{ab}$. Each part of section 2 contributes to both:

$$L_{aa} = L_m + L_{\sigma,self}, \qquad L_{ab} = -\tfrac12 L_m + L_{\sigma,mut}.$$

$L_{\sigma,mut}$ comes from the harmonic leakage and from slot and end-winding leakage where two phases share a slot or overlap at the ends.

**Synchronous (positive-sequence) inductance.** With balanced currents $i_a + i_b + i_c = 0$:

$$\lambda_a = L_{aa} i_a + L_{ab}(i_b + i_c) = (L_{aa} - L_{ab})\, i_a,$$

$$\boxed{L_s = L_{aa} - L_{ab} = \tfrac32 L_m + (L_{\sigma,self} - L_{\sigma,mut})}$$

For $m$ phases the magnetising part is $\tfrac{m}{2} L_m$. $\tfrac32 L_m$ is what induction-machine texts call the magnetising inductance $L_M$; take care which one a formula means.

**Zero sequence.** With equal currents in all phases, $L_0 = L_{aa} + 2L_{ab}$. The working harmonic cancels, so only leakage (and triplen harmonics) remain. This matters for open-end-winding drives and for checking a winding.

**dq inductances.** For a smooth rotor $L_d = L_q = L_s$. For a salient or interior-PM rotor the air-gap part depends on rotor position: $L_{aa}(\theta) \approx L_{a0} + L_{a2}\cos 2\theta$, and $L_d \ne L_q$. The leakage parts barely change with rotor position. This module computes the smooth-rotor value. A surface-PM rotor is handled by enlarging the effective gap to $k_C(g + h_m/\mu_r)$ (the magnet behaves almost like air).

**Unbalanced windings.** When the phases are not identical, $\mathbf{L}$ is no longer circulant and these single numbers are not enough. `phase_imbalance` reports how far $\mathbf{L}$ is from balanced, including how much a balanced current set couples into the negative and zero sequences.

In [ ]:
import numpy as np
from emachines.inductance import StatorGeometry, SlotGeometry, airgap_inductance, phase_imbalance
from emachines.winding import WindingPattern

# The 36-slot example from the geometry notebook: air-gap part only (leakage parts come later)
stator = StatorGeometry(Q=36, r_bore=60e-3, g=0.8e-3, l_stk=100e-3, h_y=12e-3, l_ew=22e-3,
                        slot=SlotGeometry.semi_closed(b0=2.5e-3, h0=1e-3, h1=1.5e-3, bs=6e-3, hs=18e-3, t=0.5))
pat = WindingPattern.distributed(36, 4, pitch=7, turns=10, parallel_paths=2)
L, Lm = airgap_inductance(pat, stator), airgap_inductance(pat, stator, "magnetising")
Laa, Lab = L[0, 0], L[0, 1]
print(f"L_aa = {Laa*1e3:.3f} mH, L_ab = {Lab*1e3:.3f} mH (≈ -L_aa/2 + harmonic part)")
print(f"synchronous L_s = L_aa - L_ab = {(Laa-Lab)*1e3:.3f} mH;  3/2·L_m = {1.5*Lm[0,0]*1e3:.3f} mH")
print(f"line-to-line (star) L_LL = 2(L_aa - L_ab) = {2*(Laa-Lab)*1e3:.3f} mH")
print(f"zero sequence L_0 = L_aa + 2 L_ab = {(Laa+2*Lab)*1e3:.4f} mH")
assert np.isclose(phase_imbalance(L)["L_positive"], Laa - Lab)

## 4. What a measurement contains

An inductance measurement at the terminals contains **all** the parts in section 2 at once: magnetising, harmonic, slot, tooth-tip and end-winding (plus a little from the leads). What it measures depends on how the phases are connected:

| Test | Connection | Reads | For a balanced winding |
|---|---|---|---|
| Single phase | one phase, the others open | $L_{aa}$ | $L_m + L_{\sigma,self}$ |
| Line to line (star, LCR meter on two terminals) | phases a and b in series, opposing | $L_{aa} + L_{bb} - 2L_{ab}$ | $2L_s$ → $L_s = L_{LL}/2$ |
| Series aiding / opposing | a and b in series, both ways | $L_{aa}+L_{bb} \pm 2L_{ab}$ | difference / 4 $= L_{ab}$ |
| Rotor at aligned / quadrature position | line to line, rotor locked | $2L_d$ / $2L_q$ (salient rotor) | half the min and max of $L_{LL}(\theta)$ |
| Three phases in series | a + b + c, all the same direction | $3L_0$ | leakage only |

**Effects in a measurement that the linear model leaves out**:

- **Saturation.** Inductance falls as current rises. An LCR meter drives a tiny current and sees the unsaturated value, sometimes even lower than the model, because iron permeability is low at very small flux density.
- **Permanent magnets** bias the iron. The d-axis path is partly saturated by the magnet, so $L_d$ comes out below the air-only estimate.
- **Frequency.** Eddy currents in laminations, magnets, a retaining sleeve or a rotor cage push flux out and lower the measured inductance as frequency rises. In an induction machine the cage almost shorts the air-gap flux, so a locked-rotor test measures roughly the leakage, not $L_m$.
- **Losses** show up as a series resistance that rises with frequency (iron loss, skin and proximity effect).

**Comparing model and measurement:** use a low current, a low frequency, a known rotor position (or the rotor removed), and the same connection as the formula. Expect the model to read a little high for a PM machine.

## 5. Where each part is computed

| Quantity | Symbol | Notebook | Function | Status |
|---|---|---|---|---|
| Winding layout (turns of each phase at each slot position) | $N_{x,i}$ | [Winding Patterns](winding_pattern.html) | `WindingPattern` | done |
| Stator and slot geometry | $r_{bore}, g, l_{stk}, b_0, \dots$ | [Stator and Slot Geometry](inductance_geometry.html) | `StatorGeometry`, `SlotGeometry` | done |
| Air-gap inductance (magnetising + harmonic leakage) | $L_{ag}, L_m, L_\delta$ | [Air-Gap Inductance](inductance_airgap.html) | `airgap_inductance` | done |
| Phase imbalance report | — | [Air-Gap Inductance](inductance_airgap.html) | `phase_imbalance` | done |
| Slot leakage by conductor position, tooth-tip leakage | $L_{slot}, L_{tt}$ | next notebook | — | next PR |
| End-winding leakage, phase resistance, total and dq inductances | $L_{ew}, R$ | later notebook | — | planned |
| Saturation (magnetic equivalent circuit) | — | later | — | planned |

## Symbols

| Symbol | Meaning | Unit |
|---|---|---|
| $Q$, $P$, $p$, $m$ | slots, poles, pole pairs ($P/2$), phases | — |
| $N_{x,i}$ | turns of phase $x$ in slot $i$ per ampere of phase current (signed, divided by parallel paths) | — |
| $W_{x,i}$ | winding function of phase $x$ over tooth $i$ | — |
| $\lambda$, $\Phi$ | flux linkage, flux | Wb |
| $L_{xy}$ | inductance matrix entry (self if $x = y$) | H |
| $L_m$ | magnetising inductance (working harmonic, per phase self) | H |
| $L_\delta$, $\sigma_\delta = L_\delta/L_m$ | harmonic (differential) leakage, its ratio to $L_m$ | H, — |
| $L_{slot}$, $L_{tt}$, $L_{ew}$ | slot, tooth-tip, end-winding leakage | H |
| $L_s$, $L_0$, $L_d$, $L_q$ | synchronous (positive sequence), zero sequence, d- and q-axis | H |
| $g$, $g_e = k_C g$ | mechanical and effective (Carter) air gap | m |
| $r_g = r_{bore} - g/2$ | radius at mid-gap | m |
| $l_{stk}$ | stack length | m |
| $\mu_0$ | $4\pi \times 10^{-7}$ | H/m |
| $\nu$ | harmonic order, in pole pairs round the whole gap | — |

## References

- Sudhoff, S.D. *Power Magnetic Devices* (2nd ed.), chapter 8 and [*Distributed Windings and Rotating Fields*](https://engineering.purdue.edu/~sudhoff/ee595s/TADES-chapter4.pdf), Purdue notes — winding functions, air-gap and leakage inductance.
- Pyrhönen, J., Jokinen, T., Hrabovcová, V. (2014). *Design of Rotating Electrical Machines* (2nd ed.). Wiley, chapter 4 — magnetising inductance and the leakage components (air-gap harmonic, slot, tooth-tip, end-winding, skew).
- Krause, P.C., Wasynczuk, O., Sudhoff, S.D., Pekarek, S. (2013). *Analysis of Electric Machinery and Drive Systems* (3rd ed.). Wiley-IEEE Press — phase, synchronous and dq inductances.
- IEEE Std 1812-2014, *Guide for Testing Permanent Magnet Machines* — standstill inductance tests.